# EDA — Credit-Card Transactions

**Objective.** Characterise the bank credit-card dataset (PCA-anonymised features `V1–V28` plus `Time` and `Amount`) and quantify its extreme class imbalance.

> This notebook requires `data/raw/creditcard.csv`. If the file is absent the notebook short-circuits with a clear message so the rest of the pipeline still executes.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable so `from src import ...` resolves.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

from src import config
config.ensure_dirs()
FIG = config.FIGURES_DIR


In [ ]:
from src import data_loader as dl, cleaning
from src.resampling import class_distribution

HAVE_CC = config.CREDITCARD_RAW.exists()
if not HAVE_CC:
    print('creditcard.csv not found in data/raw/ — skipping analysis.')
else:
    cc = dl.load_creditcard()
    print('Raw shape:', cc.shape)
    display(cc.head())

## 1. Overview, dtypes & data quality

In [ ]:
if HAVE_CC:
    cc.info()
    print('\nMissing values:')
    print(cleaning.missing_value_report(cc) if not cleaning.missing_value_report(cc).empty else 'None')
    print('Exact duplicate rows:', cc.duplicated().sum())

In [ ]:
if HAVE_CC:
    cc = cleaning.clean_creditcard(cc)
    print('Cleaned shape (post dedup):', cc.shape)

## 2. Class imbalance

In [ ]:
if HAVE_CC:
    dist = class_distribution(cc['Class'])
    display(dist)
    rate = cc['Class'].mean()
    print(f'Fraud rate: {rate:.4%}')
    ax = sns.countplot(x='Class', data=cc)
    ax.set(title=f'Credit-card class balance (fraud={rate:.3%})', yscale='log')
    plt.savefig(FIG / 'cc_class_balance.png', dpi=120, bbox_inches='tight')
    plt.show()

## 3. Univariate: Amount and Time

In [ ]:
if HAVE_CC:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(cc['Amount'], bins=50, ax=axes[0])
    axes[0].set(title='Transaction amount', yscale='log')
    sns.histplot(cc['Time'] / 3600, bins=48, ax=axes[1])
    axes[1].set(title='Time (hours since first txn)')
    plt.savefig(FIG / 'cc_univariate.png', dpi=120, bbox_inches='tight')
    plt.show()
    display(cc[['Amount', 'Time']].describe())

## 4. Bivariate: Amount vs class & feature correlations with target

In [ ]:
if HAVE_CC:
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.boxplot(x='Class', y='Amount', data=cc, ax=ax)
    ax.set(title='Amount by class', yscale='log')
    plt.savefig(FIG / 'cc_amount_by_class.png', dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
if HAVE_CC:
    corr = cc.corr(numeric_only=True)['Class'].drop('Class').sort_values()
    fig, ax = plt.subplots(figsize=(6, 8))
    corr.plot.barh(ax=ax)
    ax.set_title('Linear correlation of each feature with Class')
    plt.savefig(FIG / 'cc_target_correlation.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Most negatively/positively correlated PCA features:')
    print(pd.concat([corr.head(4), corr.tail(4)]))

## 5. Key findings

- **Extreme imbalance** (~0.17% fraud) — far more severe than Fraud_Data. Undersampling would discard ~99% of data; **SMOTE / SMOTEENN on the training fold only** is the appropriate response, and **AUC-PR** is the primary metric.
- `Amount` is heavily right-skewed and should be **scaled** (it is the only non-PCA numeric besides `Time`).
- Several PCA components (e.g. V14, V12, V17, V10) carry strong linear signal toward the target.